# Mon code de la competition Kaggle 2 jalon

.

##  Importation  et Configuration

In [36]:
# Imports standards
import numpy as np
import pickle
import os
import sys
from collections import Counter

# Imports PyTorch pour les modèles de deep learning
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Configuration des modèles Random Forest
# 3 variantes avec différents hyperparamètres pour diversité
MODELS_CONFIG = [
    {'name': 'Conservative', 'n_trees': 40, 'max_depth': 8,  'min_samples_split': 10},  # Plus conservateur, moins overfitting
    {'name': 'Aggressive',   'n_trees': 40, 'max_depth': 16, 'min_samples_split': 2},   # Plus agressif, capture plus de détails
    {'name': 'Balanced',     'n_trees': 40, 'max_depth': 12, 'min_samples_split': 5}    # Équilibré entre les deux
]

# Ratio de sous-échantillonnage pour accélérer l'entraînement des arbres
SUBSAMPLE_RATIO = 0.5

# Chemins des fichiers de données
PATHS = {'train': 'train_data.pkl', 'test': 'test_data.pkl'}

# Détection automatique du device (GPU si disponible, sinon CPU)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

Using device: cpu


## Data Augmentation

Augmente les données d'entraînement par des transformations géométriques.

In [37]:
def augment_batch(X):
    """
    Applique 4 transformations géométriques aux images:
    1. Image originale
    2. Flip horizontal (Left-Right)
    3. Flip vertical (Up-Down)
    4. Rotation 90°
    
    Args:
        X: Array de shape (n_samples, 28*28*3) ou (n_samples, 28, 28, 3)
    
    Returns:
        Tuple de 4 arrays transformés
    """
    n = X.shape[0]
    # Reshape en images 28x28x3 si nécessaire
    imgs = X.reshape(n, 28, 28, 3) if X.ndim == 2 else X
    
    return (
        imgs.reshape(n, -1),                              # Original
        np.flip(imgs, axis=2).reshape(n, -1),             # Flip Left-Right
        np.flip(imgs, axis=1).reshape(n, -1),             # Flip Up-Down
        np.rot90(imgs, k=1, axes=(1, 2)).reshape(n, -1)   # Rotation 90°
    )

def get_augmented_train(X, y):
    """
    Génère un dataset d'entraînement augmenté (x4 la taille originale).
    
    Args:
        X: Features d'entraînement
        y: Labels d'entraînement
    
    Returns:
        X_aug, y_aug: Données augmentées (4x plus grandes)
    """
    print(f"   [Augmentation] Génération x4...")
    x1, x2, x3, x4 = augment_batch(X)
    
    # Concatène toutes les versions augmentées
    return np.concatenate([x1, x2, x3, x4], axis=0), np.concatenate([y]*4, axis=0)

##  PyTorch Dataset

Classe pour gérer les données avec PyTorch.

In [38]:
class ImageDataset(Dataset):
    """
    Dataset PyTorch personnalisé pour les images.
    Convertit les données numpy en tensors PyTorch et normalise les pixels [0, 1].
    """
    def __init__(self, X, y=None):
        """
        Args:
            X: Images de shape (n, 28*28*3)
            y: Labels (optionnel, pour le test set)
        """
        # Reshape en (n, 3, 28, 28) format PyTorch (channels first)
        # et normalise les pixels de [0, 255] à [0, 1]
        self.X = torch.FloatTensor(X).reshape(-1, 3, 28, 28) / 255.0
        self.y = torch.LongTensor(y) if y is not None else None
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        """Retourne (image, label) ou juste image si pas de labels."""
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]

##  Modèle CNN (Convolutional Neural Network)

Réseau de neurones convolutif pour la classification d'images.

In [39]:
class CNN(nn.Module):
    """
    CNN avec 3 couches convolutionnelles.
    Architecture: Conv->BN->ReLU->Pool x3 + FC layers
    """
    def __init__(self, num_classes=5):
        super(CNN, self).__init__()
        
        # Couche convolutionnelle 
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)  # Normalisation pour stabilité
        
        # Couche convolutionnelle 
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        
        # Couche convolutionnelle 
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        
        # Pooling pour réduire les dimensions spatiales
        self.pool = nn.MaxPool2d(2, 2)  # Réduit de moitié (2x2)
        
        # Dropout pour régularisation (évite overfitting)
        self.dropout = nn.Dropout(0.5)
        
        # Couches fully connected
        # Après 3 poolings: 28->14->7->3, donc 128*3*3 = 1152 features
        self.fc1 = nn.Linear(128 * 3 * 3, 256)
        self.fc2 = nn.Linear(256, num_classes)
        
        self.relu = nn.ReLU()
        
    def forward(self, x):
        """
        Forward pass du réseau.
        
        Args:
            x: Tensor de shape (batch, 3, 28, 28)
        
        Returns:
            Logits de shape (batch, num_classes)
        """
        # Block 1: Conv->BN->ReLU->Pool (28->14)
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        
        # Block 2: Conv->BN->ReLU->Pool (14->7)
        x = self.pool(self.relu(self.bn2(self.conv2(x))))
        
        # Block 3: Conv->BN->ReLU->Pool (7->3)
        x = self.pool(self.relu(self.bn3(self.conv3(x))))
        
        # Flatten pour les couches FC
        x = x.view(x.size(0), -1)
        
        # FC layers avec dropout
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.fc2(x)
        
        return x

##  Modèle Logistic Regression (Softmax)

Modèle linéaire .

In [40]:
class LogisticRegression(nn.Module):
    """
    Régression logistique multiclasse (softmax).
    Modèle linéaire simple: y = Wx + b
    """
    def __init__(self, input_dim=28*28*3, num_classes=5):
        super(LogisticRegression, self).__init__()
        # Une seule couche linéaire
        self.linear = nn.Linear(input_dim, num_classes)
    
    def forward(self, x):
        """
        Forward pass.
        
        Args:
            x: Tensor de shape (batch, 3, 28, 28)
        
        Returns:
            Logits de shape (batch, num_classes)
        """
        # Flatten l'image en vecteur 1D
        x = x.view(x.size(0), -1)
        return self.linear(x)

##  Entraînement des Modèles PyTorch

In [41]:
def train_pytorch_model(model, train_loader, epochs=10, lr=0.001, model_name="Model"):
    """
    Entraîne un modèle PyTorch.
    
    Args:
        model: Modèle PyTorch (CNN ou LogReg)
        train_loader: DataLoader avec les données d'entraînement
        epochs: Nombre d'époques d'entraînement
        lr: Learning rate
        model_name: Nom pour l'affichage
    
    Returns:
        Modèle entraîné
    """
    # Déplacer le modèle sur GPU/CPU
    model = model.to(DEVICE)
    
    # Fonction de perte: CrossEntropy pour classification multiclasse
    criterion = nn.CrossEntropyLoss()
    
    # Optimiseur Adam (plus efficace que SGD)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    print(f"\n[{model_name}] Training for {epochs} epochs...")
    
    for epoch in range(epochs):
        model.train()  # Mode entraînement (active dropout, etc.)
        total_loss = 0
        correct = 0
        total = 0
        
        # Boucle sur les batches
        for batch_idx, (data, target) in enumerate(train_loader):
            # Déplacer les données sur GPU/CPU
            data, target = data.to(DEVICE), target.to(DEVICE)
            
            # Forward pass
            optimizer.zero_grad()  # Reset gradients
            output = model(data)
            loss = criterion(output, target)
            
            # Backward pass
            loss.backward()        # Calcul des gradients
            optimizer.step()       # Mise à jour des poids
            
            # Statistiques
            total_loss += loss.item()
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()
        
        # Affichage des métriques par époque
        acc = 100. * correct / total
        print(f"  Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f} - Acc: {acc:.2f}%")
    
    return model

##  Prédictions avec PyTorch

In [42]:
def predict_pytorch(model, X):
    """
    Génère des prédictions probabilistes avec un modèle PyTorch.
    
    Args:
        model: Modèle PyTorch entraîné
        X: Features numpy array
    
    Returns:
        Array numpy de probabilités (n_samples, n_classes)
    """
    model.eval()  # Mode évaluation (désactive dropout, etc.)
    
    # Créer un dataset et un loader
    dataset = ImageDataset(X)
    loader = DataLoader(dataset, batch_size=128, shuffle=False)
    
    all_probs = []
    
    # Désactiver le calcul des gradients (plus rapide, moins de mémoire)
    with torch.no_grad():
        for data in loader:
            # Gérer le cas où le loader retourne un tuple ou juste data
            if isinstance(data, tuple):
                data = data[0]
            
            data = data.to(DEVICE)
            output = model(data)
            
            # Convertir les logits en probabilités avec softmax
            probs = torch.softmax(output, dim=1)
            all_probs.append(probs.cpu().numpy())
    
    # Concaténer tous les batches
    return np.vstack(all_probs)

##  Random Forest (Implémentation depuis zéro)

Implémentation optimisée d'un Random Forest.

In [43]:
class Node:
    """
    Nœud d'un arbre de décision.
    Peut être soit une feuille (avec val), soit un nœud interne (avec feat, thr, left, right).
    """
    def __init__(self, val=None, feat=None, thr=None, left=None, right=None):
        self.val = val        # Valeur de prédiction (pour feuilles)
        self.feat = feat      # Index de la feature à tester
        self.thr = thr        # Seuil de décision
        self.left = left      # Sous-arbre gauche (<=)
        self.right = right    # Sous-arbre droit (>)
    
    def is_leaf(self):
        """Un nœud est une feuille s'il a une valeur de prédiction."""
        return self.val is not None

class DecisionTree:
    """
    Arbre de décision pour classification.
    Utilise l'entropie comme critère de split.
    """
    def __init__(self, max_depth, min_samples_split, n_feats):
        self.d = max_depth              # Profondeur maximale
        self.s = min_samples_split      # Minimum d'échantillons pour split
        self.n_f = n_feats              # Nombre de features à considérer
        self.root = None                # Racine de l'arbre
    
    def fit(self, X, y):
        """Entraîne l'arbre sur les données X, y."""
        self.root = self._g(X, y)
    
    def _g(self, X, y, d=0):
        """
        Construit l'arbre récursivement (grow).
        
        Args:
            X: Features
            y: Labels
            d: Profondeur actuelle
        
        Returns:
            Node (feuille ou nœud interne)
        """
        n, f = X.shape
        u = len(np.unique(y))  # Nombre de classes uniques
        
        # Conditions d'arrêt: profondeur max, une seule classe, ou trop peu d'échantillons
        if d >= self.d or u == 1 or n < self.s:
            return Node(val=self._mc(y))  # Créer une feuille
        
        # Sélection aléatoire de features (caractéristique clé du Random Forest)
        n_f_actual = min(self.n_f, f)
        f_i = np.random.choice(f, n_f_actual, replace=False)
        
        # Trouver le meilleur split
        b_f, b_t = self._bst(X, y, f_i)
        
        # Si aucun split n'est possible, créer une feuille
        if b_f is None:
            return Node(val=self._mc(y))
        
        # Split les données
        l, r = self._spl(X[:, b_f], b_t)
        
        # Si un des splits est vide, créer une feuille
        if len(l) == 0 or len(r) == 0:
            return Node(val=self._mc(y))
        
        # Créer un nœud interne et construire récursivement les sous-arbres
        return Node(
            feat=b_f,
            thr=b_t,
            left=self._g(X[l], y[l], d+1),
            right=self._g(X[r], y[r], d+1)
        )
    
    def _bst(self, X, y, f_i):
        """
        Trouve le meilleur split (best split) en maximisant le gain d'information.
        
        Args:
            X: Features
            y: Labels
            f_i: Indices des features à tester
        
        Returns:
            best_feat, best_threshold
        """
        b_g = -1       # Meilleur gain
        b_f = None     # Meilleure feature
        b_t = None     # Meilleur seuil
        
        # Tester chaque feature
        for i in f_i:
            c = X[:, i]
            th = np.unique(c)  # Valeurs uniques comme seuils candidats
            
            # Optimisation: limiter à 8 seuils max pour vitesse
            if len(th) > 8:
                th = np.random.choice(th, 8, replace=False)
            
            # Tester chaque seuil
            for t in th:
                l, r = self._spl(c, t)
                
                # Skip si un split est vide
                if len(l) == 0 or len(r) == 0:
                    continue
                
                # Calculer le gain d'information
                n = len(y)
                e_p = self._en(y)                                      # Entropie parent
                e_ch = (len(l)/n)*self._en(y[l]) + (len(r)/n)*self._en(y[r])  # Entropie enfants
                g = e_p - e_ch                                        # Gain
                
                # Garder le meilleur
                if g > b_g:
                    b_g = g
                    b_f = i
                    b_t = t
        
        return b_f, b_t
    
    def _spl(self, c, t):
        """Split une colonne selon un seuil. Retourne indices gauche et droit."""
        return np.argwhere(c <= t).flatten(), np.argwhere(c > t).flatten()
    
    def _en(self, y):
        """
        Calcule l'entropie: -sum(p * log(p))
        Mesure d'impureté (0 = pur, 1 = maximum impureté)
        """
        if len(y) == 0:
            return 0
        h = np.bincount(y)           # Compte par classe
        p = h[h > 0] / len(y)        # Probabilités
        return -np.sum(p * np.log(p + 1e-10))  # Epsilon pour éviter log(0)
    
    def _mc(self, y):
        """Retourne la classe majoritaire (most common)."""
        if len(y) == 0:
            return 0
        return np.bincount(y).argmax()
    
    def predict(self, X):
        """Prédit les classes pour X."""
        return np.array([self._trav(x, self.root) for x in X])
    
    def _trav(self, x, n):
        """
        Traverse l'arbre pour une prédiction.
        
        Args:
            x: Un échantillon
            n: Nœud actuel
        
        Returns:
            Classe prédite
        """
        # Si feuille, retourner la valeur
        if n.is_leaf():
            return n.val
        
        # Sinon, continuer à gauche ou droite selon le seuil
        if x[n.feat] <= n.thr:
            return self._trav(x, n.left)
        else:
            return self._trav(x, n.right)

In [44]:
class RandomForest:
    """
    Random Forest: Ensemble d'arbres de décision.
    Chaque arbre est entraîné sur un sous-ensemble aléatoire des données (bootstrap).
    """
    def __init__(self, name, n_trees, max_depth, min_samples_split):
        self.name = name                        # Nom du modèle
        self.nt = n_trees                       # Nombre d'arbres
        self.md = max_depth                     # Profondeur max des arbres
        self.ms = min_samples_split             # Min samples pour split
        self.trees = []                         # Liste des arbres
    
    def fit(self, X, y):
        """
        Entraîne la forêt d'arbres.
        
        Args:
            X: Features d'entraînement
            y: Labels d'entraînement
        """
        self.trees = []
        y = y.flatten().astype(int)
        
        # Nombre de features à considérer par arbre (règle sqrt)
        nf = int(np.sqrt(X.shape[1]))
        
        # Taille du sous-échantillon pour chaque arbre
        n_samples = max(int(len(X) * SUBSAMPLE_RATIO), self.ms)
        
        print(f"   [{self.name}] Train {self.nt} trees on {n_samples} samples...")
        
        for i in range(self.nt):
            # Bootstrap: échantillonnage avec remplacement
            idx = np.random.choice(len(X), n_samples, replace=True)
            
            # Créer et entraîner un arbre
            t = DecisionTree(self.md, self.ms, nf)
            t.fit(X[idx], y[idx])
            self.trees.append(t)
            
            # Afficher la progression
            sys.stdout.write(f"\r     -> Tree {i+1}/{self.nt}")
            sys.stdout.flush()
        
        print(" OK")
    
    def predict_proba_matrix(self, X):
        """
        Prédit les probabilités pour chaque classe.
        
        Args:
            X: Features
        
        Returns:
            Array de probabilités (n_samples, 5)
        """
        # Obtenir les prédictions de tous les arbres
        preds = np.array([t.predict(X) for t in self.trees]).T
        
        # Déterminer le nombre de classes
        n_classes = max(5, int(preds.max()) + 1)
        probs = np.zeros((X.shape[0], n_classes))
        
        # Pour chaque échantillon, compter les votes de chaque arbre
        for i, row in enumerate(preds):
            counts = np.bincount(row.astype(int), minlength=n_classes)
            probs[i] = counts / self.nt  # Convertir en probabilités
        
        return probs[:, :5]  # Retourner seulement les 5 premières classes

## Nous effectuons le Chargement des Données dans cette session

In [45]:
def load_data(path, is_test=False):
    """
    Charge les données depuis un fichier pickle.
    
    Args:
        path: Chemin du fichier .pkl
        is_test: Si True, pas de labels
    
    Returns:
        X, y (ou X, None si is_test=True)
    """
    if not os.path.exists(path):
        print(f"Erreur: {path} introuvable!")
        sys.exit(1)
    
    # Charger le fichier pickle
    with open(path, 'rb') as f:
        d = pickle.load(f)
    
    # Extraire et reshape les images
    X = np.array(d['images'], dtype=np.float32).reshape(len(d['images']), -1)
    
    # Extraire les labels si disponibles
    y = np.array(d['labels']).flatten().astype(int) if not is_test else None
    
    return X, y

## On fait l'Évaluation des Modèles dans cette session

In [46]:
def evaluate_model(model_name, probs, y_true):
    """
    Évalue un modèle sur base de ses prédictions probabilistes.
    
    Args:
        model_name: Nom du modèle (pour affichage)
        probs: Probabilités prédites (n_samples, n_classes)
        y_true: Vraies labels
    
    Returns:
        Accuracy (float entre 0 et 1)
    """
    # Convertir probabilités en prédictions (classe avec proba max)
    preds = np.argmax(probs, axis=1)
    
    # Calculer l'accuracy
    acc = np.mean(preds == y_true)
    
    print(f"  [{model_name}] Accuracy: {acc*100:.2f}%")
    return acc

##  PIPELINE PRINCIPAL: Entraînement et Ensemble

Dans Cette section, on fait ces   processus:
1. Chargement des données
2. Entraînement de tous les modèles
3. Évaluation sur validation set
4. Création de l'ensemble pondéré
5. Prédictions sur test set

In [ ]:
if __name__ == "__main__":
    print("="*60)
    print("ENSEMBLE: Random Forest + CNN + Logistic Regression")
    print("="*60)
    
    # ==========================================
    # ÉTAPE 1: CHARGEMENT DES DONNÉES
    # ==========================================
    print("\n[1/5] Chargement des données...")
    X_raw, y_raw = load_data(PATHS['train'])
    print(f"  Données: {X_raw.shape[0]} images")
    
    # Split train/validation (90% train, 10% validation)
    # Important pour évaluer les performances et pondérer l'ensemble
    split_idx = int(0.9 * len(X_raw))
    X_train, y_train = X_raw[:split_idx], y_raw[:split_idx]
    X_val, y_val = X_raw[split_idx:], y_raw[split_idx:]
    
    # Augmentation des données (seulement sur le train, pas sur validation!)
    X_aug, y_aug = get_augmented_train(X_train, y_train)
    print(f"  Train augmenté: {X_aug.shape[0]} images")
    print(f"  Validation: {X_val.shape[0]} images")
    
    # ==========================================
    # ÉTAPE 2: ENTRAÎNEMENT RANDOM FORESTS
    # ==========================================
    print("\n[2/5] Entraînement Random Forests...")
    forests = []
    for cfg in MODELS_CONFIG:
        # Créer et entraîner chaque variante de Random Forest
        rf = RandomForest(cfg['name'], cfg['n_trees'], cfg['max_depth'], cfg['min_samples_split'])
        rf.fit(X_aug, y_aug)
        forests.append(rf)
    
    # ==========================================
    # ÉTAPE 3: ENTRAÎNEMENT CNN
    # ==========================================
    print("\n[3/5] Entraînement CNN...")
    # Créer le DataLoader PyTorch
    train_dataset = ImageDataset(X_aug, y_aug)
    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
    
    # Créer et entraîner le CNN
    cnn_model = CNN(num_classes=5)
    cnn_model = train_pytorch_model(cnn_model, train_loader, epochs=15, lr=0.001, model_name="CNN")
    
    # ==========================================
    # ÉTAPE 4: ENTRAÎNEMENT LOGISTIC REGRESSION
    # ==========================================
    print("\n[4/5] Entraînement Logistic Regression...")
    # Créer et entraîner le modèle de régression logistique
    logreg_model = LogisticRegression(input_dim=28*28*3, num_classes=5)
    logreg_model = train_pytorch_model(logreg_model, train_loader, epochs=20, lr=0.01, model_name="LogReg")
    
    # ==========================================
    # ÉTAPE 5: ÉVALUATION SUR VALIDATION SET
    # ==========================================
    print("\n[5/5] Évaluation sur validation set...")
    
    # --- Prédictions Random Forest ---
    # Moyenne des prédictions de tous les RF
    rf_probs = np.zeros((X_val.shape[0], 5))
    for rf in forests:
        rf_probs += rf.predict_proba_matrix(X_val)
    rf_probs /= len(forests)  # Moyenne
    rf_acc = evaluate_model("Random Forest Ensemble", rf_probs, y_val)
    
    # --- Prédictions CNN ---
    cnn_probs = predict_pytorch(cnn_model, X_val)
    cnn_acc = evaluate_model("CNN", cnn_probs, y_val)
    
    # --- Prédictions Logistic Regression ---
    logreg_probs = predict_pytorch(logreg_model, X_val)
    logreg_acc = evaluate_model("Logistic Regression", logreg_probs, y_val)
    
    # ==========================================
    # SÉLECTION DU MEILLEUR MODÈLE ET PONDÉRATION
    # ==========================================
    print("\n" + "="*60)
    print("SÉLECTION DU MEILLEUR MODÈLE ET CRÉATION DE L'ENSEMBLE")
    print("="*60)
    
    # Dictionnaire des performances
    model_accs = {
        'Random Forest': rf_acc,
        'CNN': cnn_acc,
        'Logistic Regression': logreg_acc
    }
    
    # Identifier le meilleur modèle individuel
    best_model = max(model_accs, key=model_accs.get)
    print(f"\nMeilleur modèle individuel: {best_model} ({model_accs[best_model]*100:.2f}%)")
    
    # Calcul des poids proportionnels aux performances
    # Meilleur modèle aura plus de poids dans l'ensemble
    total_acc = sum(model_accs.values())
    weights = {k: v/total_acc for k, v in model_accs.items()}
    
    print(f"\nPoids de l'ensemble:")
    print(f"  Random Forest: {weights['Random Forest']:.3f}")
    print(f"  CNN: {weights['CNN']:.3f}")
    print(f"  Logistic Regression: {weights['Logistic Regression']:.3f}")
    
    # Combinaison pondérée des prédictions
    ensemble_probs = (
        weights['Random Forest'] * rf_probs + 
        weights['CNN'] * cnn_probs + 
        weights['Logistic Regression'] * logreg_probs
    )
    ensemble_acc = evaluate_model("WEIGHTED ENSEMBLE", ensemble_probs, y_val)
    
    # ==========================================
    # PRÉDICTIONS SUR LE TEST SET
    # ==========================================
    if os.path.exists(PATHS['test']):
        print("\n" + "="*60)
        print("PRÉDICTIONS SUR LE TEST SET")
        print("="*60)
        
        # Charger le test set
        X_test, _ = load_data(PATHS['test'], is_test=True)
        print(f"Test set: {X_test.shape[0]} images")
        
        # Test Time Augmentation (TTA)
        # Applique les mêmes transformations qu'au train pour plus de robustesse
        print("\nApplying Test Time Augmentation...")
        vars_tta = augment_batch(X_test)
        
        # Initialiser les probabilités finales
        final_probs = np.zeros((X_test.shape[0], 5))
        
        # --- Random Forest avec TTA ---
        print("  Random Forest predictions...")
        for rf in forests:
            for X_var in vars_tta:
                # Pondération: weight_RF / (nombre_RF * nombre_variations)
                final_probs += weights['Random Forest'] * rf.predict_proba_matrix(X_var) / (len(forests) * len(vars_tta))
        
        # --- CNN avec TTA ---
        print("  CNN predictions...")
        for X_var in vars_tta:
            final_probs += weights['CNN'] * predict_pytorch(cnn_model, X_var) / len(vars_tta)
        
        # --- Logistic Regression avec TTA ---
        print("  Logistic Regression predictions...")
        for X_var in vars_tta:
            final_probs += weights['Logistic Regression'] * predict_pytorch(logreg_model, X_var) / len(vars_tta)
        
        # Convertir probabilités en prédictions finales
        preds = np.argmax(final_probs, axis=1)
        print(f"\nDistribution des prédictions: {Counter(preds)}")
        
        # ==========================================
        # SAUVEGARDE DES RÉSULTATS
        # ==========================================
        output_file = 'submission_full_ensemble.csv'
        with open(output_file, 'w') as f:
            f.write("ID,Label\n")
            for i, p in enumerate(preds):
                f.write(f"{i+1},{p}\n")
        
        print(f"\n✓ Fichier '{output_file}' créé avec succès!")
        print("="*60)
    else:
        print(f"\n[ATTENTION] Fichier test '{PATHS['test']}' introuvable.")

ENSEMBLE: Random Forest + CNN + Logistic Regression

[1/5] Chargement des données...
  Données: 1080 images
   [Augmentation] Génération x4...
  Train augmenté: 3888 images
  Validation: 108 images

[2/5] Entraînement Random Forests...
   [Conservative] Train 40 trees on 1944 samples...
     -> Tree 40/40 OK
   [Aggressive] Train 40 trees on 1944 samples...
     -> Tree 40/40 OK
   [Balanced] Train 40 trees on 1944 samples...
     -> Tree 22/40